In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# ============================================================
# Figure: Relative objective-gap distribution
# Exact-MST vs Dynamic Greedy-MST
# Authoritative 24,000-condition primary benchmark
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator


# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path(PROJECT_ROOT)

RESULTS_DIR = ROOT / "05_Results"
FIGURES_DIR = ROOT / "06_Figures"

INPUT_CSV = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)

OUTPUT_PDF = FIGURES_DIR / "fig_gap_distribution.pdf"
OUTPUT_PNG = FIGURES_DIR / "fig_gap_distribution.png"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. LOAD AUTHORITATIVE BENCHMARK
# ------------------------------------------------------------

df = pd.read_csv(INPUT_CSV)

print("Loaded:", INPUT_CSV)
print("Rows:", len(df))
print("Columns:", len(df.columns))


# ------------------------------------------------------------
# 3. NORMALIZE COLUMN NAMES FOR ROBUST DETECTION
# ------------------------------------------------------------

def normalize_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


normalized_to_original = {
    normalize_name(col): col for col in df.columns
}


# ------------------------------------------------------------
# 4. FIND STORED RELATIVE/PERCENTAGE GAP COLUMN
# ------------------------------------------------------------

preferred_gap_names = [
    "relative_objective_gap_pct",
    "relative_objective_gap_percent",
    "relative_gap_pct",
    "relative_gap_percent",
    "optimality_gap_pct",
    "optimality_gap_percent",
    "optimality_gap_percentage",
    "relative_optimality_gap_pct",
    "relative_optimality_gap_percent",
    "gap_pct",
    "gap_percent",
]

gap_col = None

for candidate in preferred_gap_names:
    if candidate in normalized_to_original:
        gap_col = normalized_to_original[candidate]
        break


# Broader automatic search if the exact stored name differs
if gap_col is None:
    possible = []

    for col in df.columns:
        n = normalize_name(col)

        if (
            "gap" in n
            and (
                "percent" in n
                or "percentage" in n
                or "pct" in n
                or "relative" in n
            )
        ):
            possible.append(col)

    if len(possible) == 1:
        gap_col = possible[0]

    elif len(possible) > 1:
        # Prefer columns explicitly mentioning relative/optimality
        ranked = sorted(
            possible,
            key=lambda x: (
                "relative" not in normalize_name(x),
                "optimality" not in normalize_name(x),
                "percent" not in normalize_name(x)
                and "pct" not in normalize_name(x),
            )
        )
        gap_col = ranked[0]


# ------------------------------------------------------------
# 5. FALLBACK: RECONSTRUCT GAP FROM EXACT/GREEDY OBJECTIVES
# ------------------------------------------------------------

if gap_col is not None:

    print("\nDetected stored gap column:")
    print("   ", gap_col)

    gap = pd.to_numeric(df[gap_col], errors="coerce")

    # If stored as fractional gap, e.g. 0.079 instead of 7.9%
    if gap.dropna().max() <= 1.5:
        print("Detected fractional gap representation.")
        print("Converting to percentage.")
        gap = 100.0 * gap

else:

    print("\nNo direct percentage-gap column detected.")
    print("Attempting reconstruction from exact/greedy objectives...")

    exact_candidates = []
    greedy_candidates = []

    for col in df.columns:
        n = normalize_name(col)

        if (
            "exact" in n
            and ("objective" in n or "cost" in n)
        ):
            exact_candidates.append(col)

        if (
            ("greedy" in n or "algorithm" in n)
            and ("objective" in n or "cost" in n)
        ):
            greedy_candidates.append(col)

    if not exact_candidates or not greedy_candidates:
        print("\nAvailable columns:")
        for col in df.columns:
            print("  ", col)

        raise KeyError(
            "Could not identify either the stored gap column "
            "or exact/greedy objective columns."
        )

    exact_col = exact_candidates[0]
    greedy_col = greedy_candidates[0]

    print("Exact objective column :", exact_col)
    print("Greedy objective column:", greedy_col)

    exact_j = pd.to_numeric(df[exact_col], errors="coerce")
    greedy_j = pd.to_numeric(df[greedy_col], errors="coerce")

    valid = exact_j > 0

    gap = pd.Series(np.nan, index=df.index, dtype=float)

    gap.loc[valid] = (
        100.0
        * (greedy_j.loc[valid] - exact_j.loc[valid])
        / exact_j.loc[valid]
    )


# ------------------------------------------------------------
# 6. CLEAN AND VERIFY
# ------------------------------------------------------------

gap = gap.replace([np.inf, -np.inf], np.nan).dropna()

N = len(gap)

mean_gap = gap.mean()
median_gap = gap.median()
max_gap = gap.max()

q95 = gap.quantile(0.95)
q99 = gap.quantile(0.99)

large_gap_rate = 100.0 * (gap > 10.0).mean()


print("\n" + "=" * 60)
print("AUTHORITATIVE GAP SUMMARY")
print("=" * 60)
print(f"N                     = {N:,}")
print(f"Mean gap              = {mean_gap:.4f}%")
print(f"Median gap            = {median_gap:.4f}%")
print(f"Maximum gap           = {max_gap:.4f}%")
print(f"Gap > 10%             = {large_gap_rate:.4f}%")
print(f"95th percentile       = {q95:.4f}%")
print(f"99th percentile       = {q99:.4f}%")
print("=" * 60)


# ------------------------------------------------------------
# 7. SANITY CHECK AGAINST FROZEN MANUSCRIPT VALUES
# ------------------------------------------------------------

expected = {
    "N": 24000,
    "mean": 7.9219,
    "median": 2.4520,
    "maximum": 99.2977,
    "large_gap_rate": 30.5792,
    "q95": 31.1948,
    "q99": 44.2029,
}

assert N == expected["N"], (
    f"Unexpected number of eligible rows: {N:,}"
)

assert np.isclose(mean_gap, expected["mean"], atol=0.01), (
    f"Mean mismatch: {mean_gap:.4f}%"
)

assert np.isclose(median_gap, expected["median"], atol=0.01), (
    f"Median mismatch: {median_gap:.4f}%"
)

assert np.isclose(max_gap, expected["maximum"], atol=0.01), (
    f"Maximum mismatch: {max_gap:.4f}%"
)

assert np.isclose(
    large_gap_rate,
    expected["large_gap_rate"],
    atol=0.01
), (
    f"Large-gap rate mismatch: {large_gap_rate:.4f}%"
)

assert np.isclose(q95, expected["q95"], atol=0.02), (
    f"95th percentile mismatch: {q95:.4f}%"
)

assert np.isclose(q99, expected["q99"], atol=0.02), (
    f"99th percentile mismatch: {q99:.4f}%"
)

print("\n✓ Benchmark validation passed.")
print("✓ Figure will be generated from the correct 24,000 rows.")


# ------------------------------------------------------------
# 8. JOURNAL-STYLE MATPLOTLIB SETTINGS
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": [
        "Times New Roman",
        "Times",
        "DejaVu Serif"
    ],
    "mathtext.fontset": "stix",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 10,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "legend.fontsize": 7.5,

    "axes.linewidth": 0.8,

    "xtick.direction": "in",
    "ytick.direction": "in",

    "xtick.major.size": 4,
    "ytick.major.size": 4,

    "xtick.minor.size": 2,
    "ytick.minor.size": 2,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
})


# ------------------------------------------------------------
# 9. CREATE SCIENTIFIC DISTRIBUTION FIGURE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.0, 4.2))


# 2-percentage-point bins from 0 to 100
bin_width = 2.0

bins = np.arange(
    0,
    max(102, np.ceil(max_gap / bin_width) * bin_width + bin_width),
    bin_width
)


# Convert histogram counts into percentage of benchmark conditions
weights = np.ones(len(gap)) * (100.0 / len(gap))

ax.hist(
    gap,
    bins=bins,
    weights=weights,
    color="#D9E6EC",
    edgecolor="#3E5965",
    linewidth=0.65,
    alpha=1.0,
    zorder=2,
)


# ------------------------------------------------------------
# 10. REFERENCE LINES
# ------------------------------------------------------------

ax.axvline(
    median_gap,
    color="#111111",
    linewidth=1.4,
    linestyle="-",
    label=f"Median = {median_gap:.2f}%",
    zorder=5,
)

ax.axvline(
    mean_gap,
    color="#444444",
    linewidth=1.4,
    linestyle="--",
    label=f"Mean = {mean_gap:.2f}%",
    zorder=5,
)

ax.axvline(
    10.0,
    color="#9B3A34",
    linewidth=1.3,
    linestyle=":",
    label="Large-gap boundary = 10%",
    zorder=5,
)

ax.axvline(
    q95,
    color="#245B78",
    linewidth=1.3,
    linestyle="-.",
    label=f"$q_{{0.95}}$ = {q95:.2f}%",
    zorder=5,
)

ax.axvline(
    q99,
    color="#245B78",
    linewidth=1.3,
    linestyle=(0, (5, 2, 1, 2)),
    label=f"$q_{{0.99}}$ = {q99:.2f}%",
    zorder=5,
)


# ------------------------------------------------------------
# 11. AXES AND SCIENTIFIC FORMATTING
# ------------------------------------------------------------

ax.set_xlabel("Relative objective gap, $G_{\\mathrm{rel}}$ (%)")
ax.set_ylabel(
    f"Benchmark conditions (% per {bin_width:.0f}-point bin)"
)

ax.set_xlim(0, 100)

ax.xaxis.set_major_locator(MultipleLocator(10))
ax.xaxis.set_minor_locator(MultipleLocator(5))

ax.yaxis.set_minor_locator(MultipleLocator(1))


# Light horizontal guides only
ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.55,
    alpha=0.45,
    zorder=0,
)


# Remove unnecessary upper/right borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# ------------------------------------------------------------
# 12. SMALL SAMPLE-SIZE ANNOTATION
# ------------------------------------------------------------

ax.text(
    0.985,
    0.965,
    f"$N={N:,}$",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8.5,
)


# ------------------------------------------------------------
# 13. LEGEND
# ------------------------------------------------------------

legend = ax.legend(
    loc="upper right",
    frameon=True,
    framealpha=1.0,
    edgecolor="0.75",
    fancybox=False,
    borderpad=0.6,
    handlelength=3.0,
)

legend.get_frame().set_linewidth(0.6)


# ------------------------------------------------------------
# 14. FINAL LAYOUT
# ------------------------------------------------------------

fig.tight_layout()


# ------------------------------------------------------------
# 15. EXPORT
# ------------------------------------------------------------

fig.savefig(
    OUTPUT_PDF,
    format="pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_PNG,
    format="png",
    dpi=600,
    bbox_inches="tight",
)

plt.show()


print("\nSaved publication files:")
print("PDF :", OUTPUT_PDF)
print("PNG :", OUTPUT_PNG)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# ============================================================
# FIGURE 2: Greedy Failure Mechanisms
#
# Panel (a): all 24,000 benchmark conditions
# Panel (b): conditions with relative objective gap > 10%
#
# IMPORTANT:
# This version derives EVERYTHING directly from the
# authoritative 24,000-row primary benchmark.
#
# Output:
# 06_Figures/fig_failure_mechanisms.pdf
# ============================================================

from pathlib import Path
import re
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator


# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path(PROJECT_ROOT)

RESULTS_DIR = ROOT / "05_Results"
FIGURES_DIR = ROOT / "06_Figures"

INPUT_CSV = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)

OUTPUT_PDF = FIGURES_DIR / "fig_failure_mechanisms.pdf"
OUTPUT_PNG = FIGURES_DIR / "fig_failure_mechanisms.png"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. CHECK FILE
# ------------------------------------------------------------

if not INPUT_CSV.exists():
    print("ERROR: Primary benchmark not found:")
    print(INPUT_CSV)

    print("\nCSV files currently present in 05_Results:")
    for f in sorted(RESULTS_DIR.glob("*.csv")):
        print("  ", f.name)

    raise FileNotFoundError(INPUT_CSV)


# ------------------------------------------------------------
# 3. LOAD AUTHORITATIVE BENCHMARK
# ------------------------------------------------------------

df = pd.read_csv(INPUT_CSV)

print("Loaded:")
print(INPUT_CSV)

print("\nRows:", len(df))
print("Columns:", len(df.columns))


# ------------------------------------------------------------
# 4. NORMALIZE COLUMN NAMES
# ------------------------------------------------------------

def norm(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


normalized_columns = {
    norm(col): col
    for col in df.columns
}


def exact_column(candidates):
    """
    Find column using normalized exact matching.
    """
    for c in candidates:
        c_norm = norm(c)

        if c_norm in normalized_columns:
            return normalized_columns[c_norm]

    return None


def token_column(required_tokens, forbidden_tokens=None):
    """
    Find first column containing all required tokens
    and none of the forbidden tokens.
    """
    forbidden_tokens = forbidden_tokens or []

    for col in df.columns:
        n = norm(col)

        if (
            all(token in n for token in required_tokens)
            and not any(token in n for token in forbidden_tokens)
        ):
            return col

    return None


# ------------------------------------------------------------
# 5. DETECT RELATIVE GAP COLUMN
# ------------------------------------------------------------

gap_col = exact_column([
    "relative_objective_gap_pct",
    "relative_objective_gap_percent",
    "relative_gap_pct",
    "relative_gap_percent",
    "optimality_gap_pct",
    "optimality_gap_percent",
    "optimality_gap_percentage",
    "relative_optimality_gap_pct",
    "relative_optimality_gap_percent",
    "gap_pct",
    "gap_percent",
])

if gap_col is None:
    for col in df.columns:

        n = norm(col)

        if (
            "gap" in n
            and (
                "relative" in n
                or "percent" in n
                or "percentage" in n
                or "pct" in n
            )
        ):
            gap_col = col
            break


if gap_col is None:
    raise KeyError(
        "Could not automatically identify the relative-gap column.\n"
        "Run:\n\nprint(df.columns.tolist())\n\n"
        "and send me the output."
    )


gap = pd.to_numeric(
    df[gap_col],
    errors="coerce"
)


# Convert fractional gap to percentage if necessary
if gap.dropna().max() <= 1.5:
    gap = gap * 100.0


print("\nDetected relative-gap column:")
print("   ", gap_col)


# ------------------------------------------------------------
# 6. DETECT SELECTED-COUNT DIFFERENCE
# ------------------------------------------------------------

delta_col = exact_column([
    "selected_count_difference",
    "selected_count_diff",
    "count_difference",
    "count_diff",
    "delta_k",
    "greedy_minus_exact_count",
    "algorithm_minus_exact_count",
])


if delta_col is None:

    for col in df.columns:
        n = norm(col)

        if (
            "count" in n
            and (
                "difference" in n
                or "diff" in n
                or "delta" in n
            )
        ):
            delta_col = col
            break


if delta_col is not None:

    delta_k = pd.to_numeric(
        df[delta_col],
        errors="coerce"
    )

    print("\nDetected selected-count difference column:")
    print("   ", delta_col)

else:

    # --------------------------------------------------------
    # FALLBACK: reconstruct count difference
    # --------------------------------------------------------

    exact_count_col = None
    greedy_count_col = None

    for col in df.columns:

        n = norm(col)

        if (
            "exact" in n
            and "count" in n
            and "selected" in n
        ):
            exact_count_col = col

        if (
            ("greedy" in n or "algorithm" in n)
            and "count" in n
            and "selected" in n
        ):
            greedy_count_col = col


    if exact_count_col is None or greedy_count_col is None:

        print("\nCould not find selected-count difference.")
        print("\nColumns available:")
        for col in df.columns:
            print("  ", col)

        raise KeyError(
            "Need either a count-difference column or "
            "exact/greedy selected-count columns."
        )


    print("\nReconstructing selected-count difference from:")
    print("Exact :", exact_count_col)
    print("Greedy:", greedy_count_col)

    exact_count = pd.to_numeric(
        df[exact_count_col],
        errors="coerce"
    )

    greedy_count = pd.to_numeric(
        df[greedy_count_col],
        errors="coerce"
    )

    delta_k = greedy_count - exact_count


# ------------------------------------------------------------
# 7. DETECT IDENTICAL-SUBSET INDICATOR
# ------------------------------------------------------------

equal_col = exact_column([
    "selected_subset_equal",
    "selected_set_equal",
    "same_selected_subset",
    "same_selected_set",
    "subset_equal",
    "sets_equal",
    "identical_subset",
    "exact_subset_recovery",
    "same_subset",
])


if equal_col is None:

    for col in df.columns:

        n = norm(col)

        if (
            ("subset" in n or "set" in n)
            and (
                "equal" in n
                or "identical" in n
                or "same" in n
                or "recovery" in n
            )
            and "objective" not in n
        ):
            equal_col = col
            break


# ------------------------------------------------------------
# 8. CONVERT EQUALITY FLAG TO BOOLEAN
# ------------------------------------------------------------

def to_bool(series):

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    # Numeric flags such as 0/1
    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    if numeric.notna().mean() > 0.90:
        return numeric.fillna(0).astype(float) != 0

    # Text flags
    text = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    true_values = {
        "true",
        "1",
        "yes",
        "y",
        "identical",
        "same",
        "equal",
    }

    return text.isin(true_values)


if equal_col is not None:

    subset_equal = to_bool(
        df[equal_col]
    )

    print("\nDetected selected-subset equality column:")
    print("   ", equal_col)


# ------------------------------------------------------------
# 9. FALLBACK USING JACCARD = 1
# ------------------------------------------------------------

else:

    jaccard_col = exact_column([
        "jaccard",
        "jaccard_similarity",
        "selected_set_jaccard",
        "subset_jaccard",
    ])

    if jaccard_col is None:

        for col in df.columns:

            if "jaccard" in norm(col):
                jaccard_col = col
                break


    if jaccard_col is not None:

        jaccard = pd.to_numeric(
            df[jaccard_col],
            errors="coerce"
        )

        subset_equal = np.isclose(
            jaccard,
            1.0,
            atol=1e-12
        )

        subset_equal = pd.Series(
            subset_equal,
            index=df.index
        )

        print("\nNo equality flag found.")
        print("Using Jaccard = 1 to reconstruct exact subset equality:")
        print("   ", jaccard_col)

    else:

        raise KeyError(
            "Could not identify either an exact-subset equality "
            "field or a Jaccard-similarity field.\n\n"
            "Run:\nprint(df.columns.tolist())\n\n"
            "and send me the result."
        )


# ------------------------------------------------------------
# 10. CREATE FAILURE-MECHANISM CLASSIFICATION
# ------------------------------------------------------------

mechanism = np.full(
    len(df),
    "",
    dtype=object
)


# First: exact same subset
mechanism[
    subset_equal.to_numpy()
] = "Identical subset"


# Same cardinality, but not identical
mask_same_count = (
    ~subset_equal.to_numpy()
    & np.isclose(delta_k.to_numpy(dtype=float), 0.0)
)

mechanism[
    mask_same_count
] = "Same count, different items"


# Greedy more
mask_more = (
    ~subset_equal.to_numpy()
    & (delta_k.to_numpy(dtype=float) > 0)
)

mechanism[
    mask_more
] = "Greedy more items"


# Greedy fewer
mask_fewer = (
    ~subset_equal.to_numpy()
    & (delta_k.to_numpy(dtype=float) < 0)
)

mechanism[
    mask_fewer
] = "Greedy fewer items"


df_plot = pd.DataFrame({
    "gap": gap,
    "delta_k": delta_k,
    "subset_equal": subset_equal,
    "mechanism": mechanism,
})


# ------------------------------------------------------------
# 11. VERIFY COMPLETE CLASSIFICATION
# ------------------------------------------------------------

unclassified = (
    df_plot["mechanism"]
    .eq("")
    .sum()
)

if unclassified != 0:

    raise ValueError(
        f"{unclassified} rows could not be classified."
    )


print("\n✓ Every row received exactly one failure-mechanism label.")


# ------------------------------------------------------------
# 12. ORDER
# ------------------------------------------------------------

order = [
    "Identical subset",
    "Same count, different items",
    "Greedy more items",
    "Greedy fewer items",
]


display_labels = [
    "Identical subset",
    "Same count,\ndifferent items",
    "Greedy more\nitems",
    "Greedy fewer\nitems",
]


# ------------------------------------------------------------
# 13. OVERALL SUMMARY
# ------------------------------------------------------------

overall_counts = (
    df_plot["mechanism"]
    .value_counts()
    .reindex(order, fill_value=0)
)


overall_share = (
    100.0
    * overall_counts
    / len(df_plot)
)


overall = pd.DataFrame({
    "mechanism": order,
    "display": display_labels,
    "count": overall_counts.values,
    "share": overall_share.values,
})


# ------------------------------------------------------------
# 14. LARGE-GAP (>10%) SUMMARY
# ------------------------------------------------------------

large_df = df_plot[
    df_plot["gap"] > 10.0
].copy()


large_counts = (
    large_df["mechanism"]
    .value_counts()
    .reindex(order, fill_value=0)
)


large_share = (
    100.0
    * large_counts
    / len(large_df)
)


large = pd.DataFrame({
    "mechanism": order,
    "display": display_labels,
    "count": large_counts.values,
    "share": large_share.values,
})


# ------------------------------------------------------------
# 15. PRINT RECONSTRUCTED VALUES
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("ALL 24,000 CONDITIONS")
print("=" * 72)

print(
    overall[
        ["mechanism", "count", "share"]
    ].to_string(
        index=False,
        formatters={
            "share": lambda x: f"{x:.4f}%"
        }
    )
)


print("\n" + "=" * 72)
print("RELATIVE OBJECTIVE GAP > 10%")
print("=" * 72)

print(
    large[
        ["mechanism", "count", "share"]
    ].to_string(
        index=False,
        formatters={
            "share": lambda x: f"{x:.4f}%"
        }
    )
)


# ------------------------------------------------------------
# 16. AUTHORITATIVE VALIDATION
# ------------------------------------------------------------

expected_overall_counts = np.array([
    9852,
    7355,
    6760,
    33
])

expected_overall_share = np.array([
    41.0500,
    30.6458,
    28.1667,
    0.1375
])

expected_large_counts = np.array([
    0,
    1426,
    5909,
    4
])

expected_large_share = np.array([
    0.0000,
    19.4304,
    80.5151,
    0.0545
])


assert len(df_plot) == 24000, (
    f"Expected 24,000 rows but found {len(df_plot):,}"
)

assert len(large_df) == 7339, (
    f"Expected 7,339 large-gap rows but found {len(large_df):,}"
)

assert np.array_equal(
    overall["count"].to_numpy(dtype=int),
    expected_overall_counts
), (
    "Overall counts do not reproduce the frozen benchmark."
)

assert np.allclose(
    overall["share"].to_numpy(),
    expected_overall_share,
    atol=0.01
), (
    "Overall percentages do not reproduce the frozen benchmark."
)

assert np.array_equal(
    large["count"].to_numpy(dtype=int),
    expected_large_counts
), (
    "Large-gap counts do not reproduce the frozen benchmark."
)

assert np.allclose(
    large["share"].to_numpy(),
    expected_large_share,
    atol=0.01
), (
    "Large-gap percentages do not reproduce the frozen benchmark."
)


print("\n✓ Primary benchmark rows       = 24,000")
print("✓ Large-gap rows              = 7,339")
print("✓ Overall mechanism audit     = PASSED")
print("✓ Large-gap attribution audit = PASSED")


# ------------------------------------------------------------
# 17. JOURNAL FIGURE SETTINGS
# ------------------------------------------------------------

plt.rcParams.update({

    "font.family": "serif",

    "font.serif": [
        "Times New Roman",
        "Times",
        "DejaVu Serif"
    ],

    "mathtext.fontset": "stix",

    "font.size": 9,

    "axes.labelsize": 9.5,
    "axes.titlesize": 10,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "axes.linewidth": 0.8,

    "xtick.direction": "in",
    "ytick.direction": "in",

    "xtick.major.size": 4,
    "ytick.major.size": 4,

    "xtick.minor.size": 2,
    "ytick.minor.size": 2,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
})


# ------------------------------------------------------------
# 18. SCIENTIFIC COLOR PALETTE
# ------------------------------------------------------------

colors = [
    "#557A95",   # identical
    "#93A9B5",   # same-count different
    "#B55A50",   # more items
    "#AAAAAA",   # fewer items
]


# ------------------------------------------------------------
# 19. CREATE TWO-PANEL FIGURE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(8.8, 3.8),
    sharex=True
)

ax1, ax2 = axes

y = np.arange(len(order))


# ------------------------------------------------------------
# PANEL A
# ------------------------------------------------------------

bars1 = ax1.barh(
    y,
    overall["share"],
    height=0.62,
    color=colors,
    edgecolor="0.25",
    linewidth=0.65,
    zorder=3
)

ax1.set_yticks(y)
ax1.set_yticklabels(display_labels)
ax1.invert_yaxis()

ax1.set_title(
    "(a) All benchmark conditions",
    loc="left",
    fontweight="bold"
)

ax1.set_xlabel(
    "Share of conditions (%)"
)


# ------------------------------------------------------------
# PANEL B
# ------------------------------------------------------------

bars2 = ax2.barh(
    y,
    large["share"],
    height=0.62,
    color=colors,
    edgecolor="0.25",
    linewidth=0.65,
    zorder=3
)

ax2.set_yticks(y)
ax2.set_yticklabels(display_labels)
ax2.invert_yaxis()

ax2.set_title(
    "(b) Relative objective gap $>10\\%$",
    loc="left",
    fontweight="bold"
)

ax2.set_xlabel(
    "Share of large-gap conditions (%)"
)


# ------------------------------------------------------------
# 20. COMMON AXIS FORMAT
# ------------------------------------------------------------

for ax in axes:

    ax.set_xlim(0, 100)

    ax.xaxis.set_major_locator(
        MultipleLocator(20)
    )

    ax.xaxis.set_minor_locator(
        MultipleLocator(10)
    )

    ax.grid(
        axis="x",
        linestyle=":",
        linewidth=0.55,
        alpha=0.45,
        zorder=0
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ------------------------------------------------------------
# 21. LABEL BARS
# ------------------------------------------------------------

def annotate(
    ax,
    bars,
    percentages,
    counts
):

    for bar, pct, count in zip(
        bars,
        percentages,
        counts
    ):

        width = bar.get_width()

        y_mid = (
            bar.get_y()
            + bar.get_height() / 2
        )

        # Very tiny bars: put text farther right
        if width < 2:
            xpos = max(width + 1.5, 2.0)
            ha = "left"

        elif width > 88:
            xpos = width - 2
            ha = "right"

        else:
            xpos = width + 1.5
            ha = "left"

        ax.text(
            xpos,
            y_mid,
            f"{pct:.2f}%\n$n={int(count):,}$",
            va="center",
            ha=ha,
            fontsize=7.5,
            linespacing=1.05,
            clip_on=False
        )


annotate(
    ax1,
    bars1,
    overall["share"],
    overall["count"]
)

annotate(
    ax2,
    bars2,
    large["share"],
    large["count"]
)


# ------------------------------------------------------------
# 22. SAMPLE SIZE
# ------------------------------------------------------------

ax1.text(
    0.97,
    0.035,
    "$N=24{,}000$",
    transform=ax1.transAxes,
    ha="right",
    va="bottom",
    fontsize=8.2
)

ax2.text(
    0.97,
    0.035,
    "$N=7{,}339$",
    transform=ax2.transAxes,
    ha="right",
    va="bottom",
    fontsize=8.2
)


# ------------------------------------------------------------
# 23. LAYOUT
# ------------------------------------------------------------

fig.subplots_adjust(
    left=0.19,
    right=0.985,
    bottom=0.17,
    top=0.88,
    wspace=0.34
)


# ------------------------------------------------------------
# 24. EXPORT
# ------------------------------------------------------------

fig.savefig(
    OUTPUT_PDF,
    format="pdf",
    bbox_inches="tight"
)

fig.savefig(
    OUTPUT_PNG,
    format="png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


print("\nFigure successfully saved:")
print("PDF :", OUTPUT_PDF)
print("PNG :", OUTPUT_PNG)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# ============================================================
# FIGURE 3: Threshold Sensitivity
#
# Panel (a):
#   Mean and median relative objective gap vs threshold
#
# Panel (b):
#   Identical-subset recovery
#   Greedy-more-items rate
#   Relative gap > 10% rate
#
# Authoritative source:
# coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv
#
# Output:
# 06_Figures/fig_threshold_sensitivity.pdf
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter


# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path(PROJECT_ROOT)

RESULTS_DIR = ROOT / "05_Results"
FIGURES_DIR = ROOT / "06_Figures"

INPUT_CSV = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)

OUTPUT_PDF = (
    FIGURES_DIR
    / "fig_threshold_sensitivity.pdf"
)

OUTPUT_PNG = (
    FIGURES_DIR
    / "fig_threshold_sensitivity.png"
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. CHECK AUTHORITATIVE FILE
# ------------------------------------------------------------

if not INPUT_CSV.exists():

    print("ERROR: Authoritative benchmark was not found:")
    print(INPUT_CSV)

    print("\nCSV files currently present in 05_Results:")

    for f in sorted(RESULTS_DIR.glob("*.csv")):
        print("  ", f.name)

    raise FileNotFoundError(INPUT_CSV)


# ------------------------------------------------------------
# 3. LOAD BENCHMARK
# ------------------------------------------------------------

df = pd.read_csv(INPUT_CSV)

print("Loaded:")
print(INPUT_CSV)

print("\nRows:", len(df))
print("Columns:", len(df.columns))


# ------------------------------------------------------------
# 4. COLUMN-NAME HELPERS
# ------------------------------------------------------------

def norm(text):
    """
    Normalize column names for robust matching.
    """
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


normalized_columns = {
    norm(col): col
    for col in df.columns
}


def exact_column(candidates):
    """
    Search by normalized exact name.
    """
    for candidate in candidates:

        candidate_norm = norm(candidate)

        if candidate_norm in normalized_columns:
            return normalized_columns[candidate_norm]

    return None


# ------------------------------------------------------------
# 5. DETECT THRESHOLD COLUMN
# ------------------------------------------------------------

threshold_col = exact_column([
    "threshold",
    "tau",
    "sufficiency_threshold",
    "modeled_sufficiency_threshold",
    "utility_threshold",
    "mst_threshold",
])


if threshold_col is None:

    for col in df.columns:

        n = norm(col)

        if (
            "threshold" in n
            or n == "tau"
        ):
            threshold_col = col
            break


if threshold_col is None:

    print("\nAvailable columns:")
    print(df.columns.tolist())

    raise KeyError(
        "Could not automatically detect the threshold column."
    )


threshold = pd.to_numeric(
    df[threshold_col],
    errors="coerce"
)

print("\nDetected threshold column:")
print("   ", threshold_col)


# ------------------------------------------------------------
# 6. DETECT RELATIVE OBJECTIVE-GAP COLUMN
# ------------------------------------------------------------

gap_col = exact_column([
    "relative_objective_gap_pct",
    "relative_objective_gap_percent",
    "relative_gap_pct",
    "relative_gap_percent",
    "optimality_gap_pct",
    "optimality_gap_percent",
    "optimality_gap_percentage",
    "relative_optimality_gap_pct",
    "relative_optimality_gap_percent",
    "gap_pct",
    "gap_percent",
])


if gap_col is None:

    for col in df.columns:

        n = norm(col)

        if (
            "gap" in n
            and (
                "relative" in n
                or "percent" in n
                or "percentage" in n
                or "pct" in n
            )
        ):
            gap_col = col
            break


if gap_col is None:

    raise KeyError(
        "Could not automatically detect the relative-gap column."
    )


gap = pd.to_numeric(
    df[gap_col],
    errors="coerce"
)


# Convert fraction -> percentage if needed
if gap.dropna().max() <= 1.5:
    gap = 100.0 * gap


print("\nDetected relative-gap column:")
print("   ", gap_col)


# ------------------------------------------------------------
# 7. DETECT SELECTED-COUNT DIFFERENCE
# ------------------------------------------------------------

delta_col = exact_column([
    "selected_count_difference",
    "selected_count_diff",
    "count_difference",
    "count_diff",
    "delta_k",
    "greedy_minus_exact_count",
    "algorithm_minus_exact_count",
])


if delta_col is None:

    for col in df.columns:

        n = norm(col)

        if (
            "count" in n
            and (
                "difference" in n
                or "diff" in n
                or "delta" in n
            )
        ):
            delta_col = col
            break


if delta_col is not None:

    delta_k = pd.to_numeric(
        df[delta_col],
        errors="coerce"
    )

    print("\nDetected count-difference column:")
    print("   ", delta_col)


else:

    # --------------------------------------------------------
    # FALLBACK: reconstruct from exact and greedy counts
    # --------------------------------------------------------

    exact_count_col = None
    greedy_count_col = None

    for col in df.columns:

        n = norm(col)

        if (
            "exact" in n
            and "selected" in n
            and "count" in n
        ):
            exact_count_col = col

        if (
            ("greedy" in n or "algorithm" in n)
            and "selected" in n
            and "count" in n
        ):
            greedy_count_col = col


    if (
        exact_count_col is None
        or greedy_count_col is None
    ):

        raise KeyError(
            "Could not identify selected-count difference "
            "or exact/greedy selected-count columns."
        )


    exact_count = pd.to_numeric(
        df[exact_count_col],
        errors="coerce"
    )

    greedy_count = pd.to_numeric(
        df[greedy_count_col],
        errors="coerce"
    )

    delta_k = greedy_count - exact_count

    print("\nReconstructed count difference from:")
    print("Exact :", exact_count_col)
    print("Greedy:", greedy_count_col)


# ------------------------------------------------------------
# 8. DETECT IDENTICAL-SUBSET INDICATOR
# ------------------------------------------------------------

equal_col = exact_column([
    "selected_subset_equal",
    "selected_set_equal",
    "same_selected_subset",
    "same_selected_set",
    "subset_equal",
    "sets_equal",
    "identical_subset",
    "exact_subset_recovery",
    "same_subset",
])


if equal_col is None:

    for col in df.columns:

        n = norm(col)

        if (
            ("subset" in n or "set" in n)
            and (
                "equal" in n
                or "identical" in n
                or "same" in n
                or "recovery" in n
            )
            and "objective" not in n
        ):
            equal_col = col
            break


def to_bool(series):
    """
    Robust conversion of stored equality flags.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    if numeric.notna().mean() > 0.90:

        return (
            numeric
            .fillna(0)
            .astype(float)
            != 0
        )

    text = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return text.isin({
        "true",
        "1",
        "yes",
        "y",
        "identical",
        "same",
        "equal",
    })


if equal_col is not None:

    subset_equal = to_bool(
        df[equal_col]
    )

    print("\nDetected subset-equality column:")
    print("   ", equal_col)


else:

    # --------------------------------------------------------
    # FALLBACK: Jaccard == 1
    # --------------------------------------------------------

    jaccard_col = exact_column([
        "jaccard",
        "jaccard_similarity",
        "selected_set_jaccard",
        "subset_jaccard",
    ])


    if jaccard_col is None:

        for col in df.columns:

            if "jaccard" in norm(col):
                jaccard_col = col
                break


    if jaccard_col is None:

        raise KeyError(
            "Could not identify either subset equality "
            "or Jaccard similarity."
        )


    jaccard = pd.to_numeric(
        df[jaccard_col],
        errors="coerce"
    )

    subset_equal = pd.Series(
        np.isclose(
            jaccard,
            1.0,
            atol=1e-12
        ),
        index=df.index
    )

    print("\nUsing Jaccard=1 to reconstruct identical subsets:")
    print("   ", jaccard_col)


# ------------------------------------------------------------
# 9. BUILD ANALYSIS DATAFRAME
# ------------------------------------------------------------

analysis = pd.DataFrame({
    "threshold": threshold,
    "gap_pct": gap,
    "delta_k": delta_k,
    "subset_equal": subset_equal,
})


analysis = analysis.dropna(
    subset=[
        "threshold",
        "gap_pct",
        "delta_k"
    ]
)


# ------------------------------------------------------------
# 10. VERIFY THRESHOLD VALUES
# ------------------------------------------------------------

thresholds = np.array([
    0.70,
    0.80,
    0.90,
    0.95
])


observed_thresholds = np.sort(
    analysis["threshold"]
    .unique()
)


print("\nObserved threshold values:")
print(observed_thresholds)


assert np.allclose(
    observed_thresholds,
    thresholds,
    atol=1e-12
), (
    "Observed thresholds do not match "
    "{0.70, 0.80, 0.90, 0.95}."
)


# ------------------------------------------------------------
# 11. CALCULATE THRESHOLD-SPECIFIC SUMMARIES
# ------------------------------------------------------------

rows = []

for tau in thresholds:

    sub = analysis[
        np.isclose(
            analysis["threshold"],
            tau,
            atol=1e-12
        )
    ].copy()

    N = len(sub)

    mean_gap = sub["gap_pct"].mean()
    median_gap = sub["gap_pct"].median()

    identical_rate = (
        100.0
        * sub["subset_equal"].mean()
    )

    more_items_rate = (
        100.0
        * (sub["delta_k"] > 0).mean()
    )

    large_gap_rate = (
        100.0
        * (sub["gap_pct"] > 10.0).mean()
    )

    mean_delta_k = sub["delta_k"].mean()

    rows.append({
        "threshold": tau,
        "N": N,
        "mean_gap": mean_gap,
        "median_gap": median_gap,
        "identical_rate": identical_rate,
        "more_items_rate": more_items_rate,
        "large_gap_rate": large_gap_rate,
        "mean_delta_k": mean_delta_k,
    })


summary = pd.DataFrame(rows)


# ------------------------------------------------------------
# 12. PRINT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("THRESHOLD-SPECIFIC SUMMARY")
print("=" * 100)

print(
    summary.to_string(
        index=False,
        formatters={
            "threshold": lambda x: f"{x:.2f}",
            "mean_gap": lambda x: f"{x:.4f}%",
            "median_gap": lambda x: f"{x:.4f}%",
            "identical_rate": lambda x: f"{x:.4f}%",
            "more_items_rate": lambda x: f"{x:.4f}%",
            "large_gap_rate": lambda x: f"{x:.4f}%",
            "mean_delta_k": lambda x: f"{x:.4f}",
        }
    )
)


# ------------------------------------------------------------
# 13. VALIDATE AGAINST FROZEN RESULTS
# ------------------------------------------------------------

expected_N = np.array([
    6000,
    6000,
    6000,
    6000
])

expected_mean_gap = np.array([
    8.7965,
    7.9873,
    7.6493,
    7.2545
])

expected_median_gap = np.array([
    1.3292,
    2.4129,
    2.9153,
    2.8164
])

expected_identical = np.array([
    46.4333,
    41.2333,
    38.9500,
    37.5833
])

expected_more_items = np.array([
    20.5500,
    22.9500,
    32.0667,
    37.1000
])

expected_large_gap = np.array([
    29.7000,
    28.7167,
    31.4167,
    32.4833
])

expected_mean_delta = np.array([
    0.2108,
    0.2443,
    0.3620,
    0.4477
])


assert np.array_equal(
    summary["N"].to_numpy(dtype=int),
    expected_N
), "Threshold denominators do not match 6,000 each."


assert np.allclose(
    summary["mean_gap"],
    expected_mean_gap,
    atol=0.01
), "Mean-gap values do not match frozen results."


assert np.allclose(
    summary["median_gap"],
    expected_median_gap,
    atol=0.01
), "Median-gap values do not match frozen results."


assert np.allclose(
    summary["identical_rate"],
    expected_identical,
    atol=0.01
), "Identical-subset rates do not match frozen results."


assert np.allclose(
    summary["more_items_rate"],
    expected_more_items,
    atol=0.01
), "Greedy-more-items rates do not match frozen results."


assert np.allclose(
    summary["large_gap_rate"],
    expected_large_gap,
    atol=0.01
), "Large-gap rates do not match frozen results."


assert np.allclose(
    summary["mean_delta_k"],
    expected_mean_delta,
    atol=0.01
), "Mean selected-count differences do not match frozen results."


print("\n✓ Threshold denominator audit passed.")
print("✓ Mean-gap audit passed.")
print("✓ Median-gap audit passed.")
print("✓ Identical-subset audit passed.")
print("✓ Greedy-over-selection audit passed.")
print("✓ Large-gap audit passed.")
print("✓ Selected-count audit passed.")


# ------------------------------------------------------------
# 14. JOURNAL-STYLE SETTINGS
# ------------------------------------------------------------

plt.rcParams.update({

    "font.family": "serif",

    "font.serif": [
        "Times New Roman",
        "Times",
        "DejaVu Serif"
    ],

    "mathtext.fontset": "stix",

    "font.size": 9,

    "axes.labelsize": 9.5,
    "axes.titlesize": 10,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "legend.fontsize": 8,

    "axes.linewidth": 0.8,

    "xtick.direction": "in",
    "ytick.direction": "in",

    "xtick.major.size": 4,
    "ytick.major.size": 4,

    "xtick.minor.size": 2,
    "ytick.minor.size": 2,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
})


# ------------------------------------------------------------
# 15. DATA FOR PLOT
# ------------------------------------------------------------

x = summary["threshold"].to_numpy()

mean_gap = summary["mean_gap"].to_numpy()
median_gap = summary["median_gap"].to_numpy()

identical = summary["identical_rate"].to_numpy()
more_items = summary["more_items_rate"].to_numpy()
large_gap = summary["large_gap_rate"].to_numpy()


# ------------------------------------------------------------
# 16. CREATE TWO-PANEL FIGURE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(8.8, 3.8)
)

ax1, ax2 = axes


# ------------------------------------------------------------
# PANEL (a): MEAN / MEDIAN GAP
# ------------------------------------------------------------

ax1.plot(
    x,
    mean_gap,
    marker="o",
    markersize=5.5,
    linewidth=1.5,
    color="#315B73",
    label="Mean relative gap"
)

ax1.plot(
    x,
    median_gap,
    marker="s",
    markersize=5.0,
    linewidth=1.4,
    linestyle="--",
    color="#9A5A45",
    label="Median relative gap"
)


ax1.set_title(
    "(a) Relative objective gap",
    loc="left",
    fontweight="bold"
)

ax1.set_xlabel(
    "Modeled sufficiency threshold, $\\tau$"
)

ax1.set_ylabel(
    "Relative objective gap (%)"
)

ax1.set_xticks(x)

ax1.set_xticklabels([
    "0.70",
    "0.80",
    "0.90",
    "0.95"
])

ax1.set_xlim(
    0.675,
    0.975
)

ax1.set_ylim(
    0,
    10
)

ax1.yaxis.set_major_locator(
    MultipleLocator(2)
)

ax1.yaxis.set_minor_locator(
    MultipleLocator(1)
)


# ------------------------------------------------------------
# LABEL PANEL A POINTS
# ------------------------------------------------------------

for xi, yi in zip(x, mean_gap):

    ax1.annotate(
        f"{yi:.2f}",
        (xi, yi),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=7.4
    )


for xi, yi in zip(x, median_gap):

    ax1.annotate(
        f"{yi:.2f}",
        (xi, yi),
        xytext=(0, -10),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=7.4
    )


ax1.legend(
    loc="center right",
    frameon=True,
    framealpha=1.0,
    edgecolor="0.75",
    fancybox=False
)


# ------------------------------------------------------------
# PANEL (b): STRUCTURAL OUTCOME RATES
# ------------------------------------------------------------

ax2.plot(
    x,
    identical,
    marker="o",
    markersize=5.5,
    linewidth=1.5,
    color="#315B73",
    label="Identical subset"
)

ax2.plot(
    x,
    more_items,
    marker="^",
    markersize=5.5,
    linewidth=1.5,
    linestyle="--",
    color="#B05045",
    label="Greedy more items"
)

ax2.plot(
    x,
    large_gap,
    marker="s",
    markersize=5.0,
    linewidth=1.4,
    linestyle="-.",
    color="#5F6B6D",
    label="$G_{\\mathrm{rel}}>10\\%$"
)


ax2.set_title(
    "(b) Structural outcome rates",
    loc="left",
    fontweight="bold"
)

ax2.set_xlabel(
    "Modeled sufficiency threshold, $\\tau$"
)

ax2.set_ylabel(
    "Benchmark conditions (%)"
)

ax2.set_xticks(x)

ax2.set_xticklabels([
    "0.70",
    "0.80",
    "0.90",
    "0.95"
])

ax2.set_xlim(
    0.675,
    0.975
)

ax2.set_ylim(
    15,
    50
)

ax2.yaxis.set_major_locator(
    MultipleLocator(5)
)

ax2.yaxis.set_minor_locator(
    MultipleLocator(2.5)
)


# ------------------------------------------------------------
# LABEL PANEL B POINTS
# ------------------------------------------------------------

for xi, yi in zip(x, identical):

    ax2.annotate(
        f"{yi:.1f}",
        (xi, yi),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=7.2
    )


for xi, yi in zip(x, more_items):

    ax2.annotate(
        f"{yi:.1f}",
        (xi, yi),
        xytext=(0, -10),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=7.2
    )


# Large-gap labels moved slightly right/down to avoid overlap
offsets = [
    (6, 3),
    (6, -10),
    (6, 4),
    (6, 4),
]

for xi, yi, offset in zip(
    x,
    large_gap,
    offsets
):

    ax2.annotate(
        f"{yi:.1f}",
        (xi, yi),
        xytext=offset,
        textcoords="offset points",
        ha="left",
        fontsize=7.2
    )


ax2.legend(
    loc="best",
    frameon=True,
    framealpha=1.0,
    edgecolor="0.75",
    fancybox=False
)


# ------------------------------------------------------------
# 17. COMMON SCIENTIFIC FORMATTING
# ------------------------------------------------------------

for ax in axes:

    ax.grid(
        axis="y",
        linestyle=":",
        linewidth=0.55,
        alpha=0.45,
        zorder=0
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        which="both",
        direction="in"
    )


# ------------------------------------------------------------
# 18. SAMPLE-SIZE ANNOTATIONS
# ------------------------------------------------------------

ax1.text(
    0.98,
    0.035,
    "$N=6{,}000$ per threshold",
    transform=ax1.transAxes,
    ha="right",
    va="bottom",
    fontsize=8
)

ax2.text(
    0.98,
    0.035,
    "$N=6{,}000$ per threshold",
    transform=ax2.transAxes,
    ha="right",
    va="bottom",
    fontsize=8
)


# ------------------------------------------------------------
# 19. FINAL LAYOUT
# ------------------------------------------------------------

fig.subplots_adjust(
    left=0.085,
    right=0.985,
    bottom=0.18,
    top=0.89,
    wspace=0.28
)


# ------------------------------------------------------------
# 20. EXPORT
# ------------------------------------------------------------

fig.savefig(
    OUTPUT_PDF,
    format="pdf",
    bbox_inches="tight"
)

fig.savefig(
    OUTPUT_PNG,
    format="png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


print("\nFigure successfully saved:")
print("PDF :", OUTPUT_PDF)
print("PNG :", OUTPUT_PNG)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# ============================================================
# FIGURE 4: Dynamic Greedy-MST Scalability
#
# Panel (a):
#   Median matched-group runtime vs candidate-pool size
#   + empirical power-law fit
#
# Panel (b):
#   Mean selected count
#   + mean selected fraction
#
# Authoritative source:
# 05_Results/greedy_scalability_v2.csv
#
# Output:
# 06_Figures/fig_scalability.pdf
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter


# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path(PROJECT_ROOT)

RESULTS_DIR = ROOT / "05_Results"
FIGURES_DIR = ROOT / "06_Figures"

INPUT_CSV = RESULTS_DIR / "greedy_scalability_v2.csv"

OUTPUT_PDF = FIGURES_DIR / "fig_scalability.pdf"
OUTPUT_PNG = FIGURES_DIR / "fig_scalability.png"

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. CHECK INPUT FILE
# ------------------------------------------------------------

if not INPUT_CSV.exists():

    print("ERROR: Scalability file not found:")
    print(INPUT_CSV)

    print("\nCSV files currently available in 05_Results:")

    for f in sorted(RESULTS_DIR.glob("*.csv")):
        print("  ", f.name)

    raise FileNotFoundError(INPUT_CSV)


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(INPUT_CSV)

print("Loaded:")
print(INPUT_CSV)

print("\nRows   :", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
for col in df.columns:
    print("  ", col)


# ------------------------------------------------------------
# 4. COLUMN-NAME HELPERS
# ------------------------------------------------------------

def norm(text):
    """
    Normalize a column name for robust matching.
    """
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


normalized_columns = {
    norm(col): col
    for col in df.columns
}


def exact_column(candidates):
    """
    Find a column by normalized exact name.
    """
    for candidate in candidates:

        n = norm(candidate)

        if n in normalized_columns:
            return normalized_columns[n]

    return None


# ------------------------------------------------------------
# 5. DETECT CANDIDATE-POOL SIZE
# ------------------------------------------------------------

size_col = exact_column([
    "candidate_set_size",
    "candidate_pool_size",
    "pool_size",
    "n_items",
    "num_items",
    "n",
    "m",
    "size",
])


if size_col is None:

    for col in df.columns:

        n = norm(col)

        if (
            "candidate" in n
            and "size" in n
        ):
            size_col = col
            break


if size_col is None:

    raise KeyError(
        "Could not detect candidate-set-size column."
    )


M = pd.to_numeric(
    df[size_col],
    errors="coerce"
)

print("\nDetected candidate-size column:")
print("   ", size_col)


# ------------------------------------------------------------
# 6. DETECT MATCHED-GROUP / REPETITION COLUMN
# ------------------------------------------------------------

group_col = exact_column([
    "matched_group_id",
    "matched_group",
    "group_id",
    "group",
    "repetition",
    "replicate",
    "rep",
    "repeat",
])


if group_col is None:

    for col in df.columns:

        n = norm(col)

        if (
            "group" in n
            or "repetition" in n
            or "replicate" in n
        ):
            group_col = col
            break


if group_col is None:

    raise KeyError(
        "Could not identify the matched-group/repetition column. "
        "This column is required because the reported runtime is "
        "the median of matched-group median runtimes."
    )


print("\nDetected matched-group column:")
print("   ", group_col)


# ------------------------------------------------------------
# 7. DETECT RUNTIME COLUMN
# ------------------------------------------------------------

runtime_col = exact_column([
    "runtime_ms",
    "greedy_runtime_ms",
    "algorithm_runtime_ms",
    "runtime_milliseconds",
    "runtime_sec",
    "runtime_seconds",
    "greedy_runtime_sec",
    "greedy_runtime_seconds",
    "runtime_s",
    "runtime",
])


if runtime_col is None:

    possible = []

    for col in df.columns:

        n = norm(col)

        if (
            "runtime" in n
            or "time_ms" in n
            or "time_sec" in n
        ):
            possible.append(col)

    if len(possible) == 1:
        runtime_col = possible[0]

    elif len(possible) > 1:

        # Prefer explicitly greedy runtime
        greedy_candidates = [
            col for col in possible
            if "greedy" in norm(col)
        ]

        if len(greedy_candidates) == 1:
            runtime_col = greedy_candidates[0]


if runtime_col is None:

    raise KeyError(
        "Could not automatically detect the greedy runtime column."
    )


runtime = pd.to_numeric(
    df[runtime_col],
    errors="coerce"
)


# ------------------------------------------------------------
# 8. CONVERT RUNTIME TO MILLISECONDS
# ------------------------------------------------------------

runtime_name = norm(runtime_col)

if (
    "millisecond" in runtime_name
    or runtime_name.endswith("_ms")
    or "_ms_" in runtime_name
):
    runtime_ms = runtime.copy()

elif (
    "second" in runtime_name
    or runtime_name.endswith("_sec")
    or runtime_name.endswith("_s")
):
    runtime_ms = runtime * 1000.0

else:
    # Infer unit carefully from magnitude.
    #
    # Expected row-level times are small:
    # if typical runtime < 0.1, it is almost certainly seconds.
    typical_runtime = runtime.dropna().median()

    if typical_runtime < 0.1:
        print(
            "\nRuntime units inferred as seconds; "
            "converting to milliseconds."
        )
        runtime_ms = runtime * 1000.0
    else:
        print(
            "\nRuntime units inferred as milliseconds."
        )
        runtime_ms = runtime.copy()


print("\nDetected runtime column:")
print("   ", runtime_col)


# ------------------------------------------------------------
# 9. DETECT SELECTED-COUNT COLUMN
# ------------------------------------------------------------

selected_count_col = exact_column([
    "selected_count",
    "selected_item_count",
    "selected_items_count",
    "num_selected",
    "n_selected",
    "greedy_selected_count",
    "algorithm_selected_count",
])


if selected_count_col is None:

    for col in df.columns:

        n = norm(col)

        if (
            "selected" in n
            and "count" in n
            and "exact" not in n
        ):
            selected_count_col = col
            break


if selected_count_col is None:

    raise KeyError(
        "Could not detect selected-count column."
    )


selected_count = pd.to_numeric(
    df[selected_count_col],
    errors="coerce"
)

print("\nDetected selected-count column:")
print("   ", selected_count_col)


# ------------------------------------------------------------
# 10. BUILD CLEAN DATAFRAME
# ------------------------------------------------------------

data = pd.DataFrame({
    "M": M,
    "group": df[group_col],
    "runtime_ms": runtime_ms,
    "selected_count": selected_count,
})


data = data.dropna(
    subset=[
        "M",
        "group",
        "runtime_ms",
        "selected_count"
    ]
)


data["M"] = data["M"].astype(int)

data["selected_fraction_pct"] = (
    100.0
    * data["selected_count"]
    / data["M"]
)


# ------------------------------------------------------------
# 11. VERIFY EXPECTED CANDIDATE SIZES
# ------------------------------------------------------------

expected_sizes = np.array([
    16,
    32,
    64,
    128,
    256
])


observed_sizes = np.sort(
    data["M"].unique()
)


print("\nObserved candidate sizes:")
print(observed_sizes)


assert np.array_equal(
    observed_sizes,
    expected_sizes
), (
    "Candidate-set sizes do not match "
    "{16, 32, 64, 128, 256}."
)


# ------------------------------------------------------------
# 12. MATCHED-GROUP RUNTIME AGGREGATION
#
# Frozen definition:
# For each M:
#   1. median runtime within each matched group
#   2. median across the 100 matched groups
# ------------------------------------------------------------

group_runtime = (
    data
    .groupby(["M", "group"], as_index=False)
    ["runtime_ms"]
    .median()
    .rename(
        columns={
            "runtime_ms": "group_median_runtime_ms"
        }
    )
)


# Count matched groups per size
group_counts = (
    group_runtime
    .groupby("M")
    ["group"]
    .nunique()
)


print("\nMatched groups per candidate size:")
print(group_counts)


assert np.all(
    group_counts.reindex(expected_sizes).values == 100
), (
    "Expected exactly 100 matched groups per candidate size."
)


runtime_summary = (
    group_runtime
    .groupby("M", as_index=False)
    ["group_median_runtime_ms"]
    .median()
    .rename(
        columns={
            "group_median_runtime_ms":
            "median_runtime_ms"
        }
    )
)


# ------------------------------------------------------------
# 13. SOLUTION-COMPACTNESS SUMMARY
# ------------------------------------------------------------

compactness_summary = (
    data
    .groupby("M", as_index=False)
    .agg(
        mean_selected_count=(
            "selected_count",
            "mean"
        ),
        mean_selected_fraction_pct=(
            "selected_fraction_pct",
            "mean"
        )
    )
)


summary = runtime_summary.merge(
    compactness_summary,
    on="M",
    how="inner"
)


# ------------------------------------------------------------
# 14. PRINT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LARGE-SIZE GREEDY SCALABILITY SUMMARY")
print("=" * 80)

print(
    summary.to_string(
        index=False,
        formatters={
            "median_runtime_ms":
                lambda x: f"{x:.4f}",
            "mean_selected_count":
                lambda x: f"{x:.4f}",
            "mean_selected_fraction_pct":
                lambda x: f"{x:.4f}%",
        }
    )
)


# ------------------------------------------------------------
# 15. VALIDATE AGAINST FROZEN RESULTS
# ------------------------------------------------------------

expected_runtime = np.array([
    0.4316,
    2.2694,
    6.0006,
    19.8935,
    63.3892
])

expected_selected_count = np.array([
    3.8596,
    3.9450,
    4.1117,
    4.4085,
    4.6558
])

expected_selected_fraction = np.array([
    24.1224,
    12.3281,
    6.4245,
    3.4442,
    1.8187
])


assert np.array_equal(
    summary["M"].to_numpy(),
    expected_sizes
)


assert np.allclose(
    summary["median_runtime_ms"],
    expected_runtime,
    atol=0.02
), (
    "Runtime summary does not reproduce "
    "the frozen scalability results."
)


assert np.allclose(
    summary["mean_selected_count"],
    expected_selected_count,
    atol=0.01
), (
    "Mean selected counts do not reproduce "
    "the frozen results."
)


assert np.allclose(
    summary["mean_selected_fraction_pct"],
    expected_selected_fraction,
    atol=0.02
), (
    "Selected fractions do not reproduce "
    "the frozen results."
)


print("\n✓ Runtime audit passed.")
print("✓ Selected-count audit passed.")
print("✓ Selected-fraction audit passed.")


# ------------------------------------------------------------
# 16. EMPIRICAL POWER-LAW FIT
#
# log2(T_M) = a + alpha log2(M)
# ------------------------------------------------------------

x = summary["M"].to_numpy(dtype=float)
y_runtime = summary["median_runtime_ms"].to_numpy(dtype=float)

log_M = np.log2(x)
log_T = np.log2(y_runtime)

alpha, intercept = np.polyfit(
    log_M,
    log_T,
    1
)

predicted_log_T = (
    intercept
    + alpha * log_M
)

predicted_T = 2 ** predicted_log_T


# R^2 on log2 scale
ss_res = np.sum(
    (log_T - predicted_log_T) ** 2
)

ss_tot = np.sum(
    (log_T - log_T.mean()) ** 2
)

r_squared = (
    1.0
    - ss_res / ss_tot
)

doubling_factor = 2 ** alpha


print("\n" + "=" * 60)
print("POWER-LAW FIT")
print("=" * 60)

print(f"alpha           = {alpha:.4f}")
print(f"doubling factor = {doubling_factor:.4f}")
print(f"R^2             = {r_squared:.4f}")


# Frozen-result audit
assert np.isclose(
    alpha,
    1.7529,
    atol=0.01
)

assert np.isclose(
    doubling_factor,
    3.3703,
    atol=0.02
)

assert np.isclose(
    r_squared,
    0.9930,
    atol=0.002
)


print("\n✓ Power-law fit audit passed.")


# ------------------------------------------------------------
# 17. SMOOTH FIT LINE FOR DISPLAY
# ------------------------------------------------------------

M_fit = np.logspace(
    np.log10(16),
    np.log10(256),
    300
)

T_fit = (
    2 ** intercept
    * M_fit ** alpha
)


# ------------------------------------------------------------
# 18. JOURNAL STYLE
# ------------------------------------------------------------

plt.rcParams.update({

    "font.family": "serif",

    "font.serif": [
        "Times New Roman",
        "Times",
        "DejaVu Serif"
    ],

    "mathtext.fontset": "stix",

    "font.size": 9,

    "axes.labelsize": 9.5,
    "axes.titlesize": 10,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "legend.fontsize": 7.8,

    "axes.linewidth": 0.8,

    "xtick.direction": "in",
    "ytick.direction": "in",

    "xtick.major.size": 4,
    "ytick.major.size": 4,

    "xtick.minor.size": 2,
    "ytick.minor.size": 2,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
})


# ------------------------------------------------------------
# 19. CREATE FIGURE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(8.8, 3.8)
)

ax1, ax2 = axes


# ============================================================
# PANEL (a): RUNTIME SCALING
# ============================================================

ax1.plot(
    x,
    y_runtime,
    marker="o",
    markersize=5.5,
    linewidth=1.5,
    color="#315B73",
    label="Observed median runtime",
    zorder=4
)

ax1.plot(
    M_fit,
    T_fit,
    linestyle="--",
    linewidth=1.3,
    color="#9A5A45",
    label=(
        rf"Power-law fit: $T\propto M^{{{alpha:.3f}}}$"
    ),
    zorder=3
)


# Log-log axes are scientifically appropriate here
ax1.set_xscale("log", base=2)
ax1.set_yscale("log")


ax1.set_xticks(
    expected_sizes
)

ax1.set_xticklabels([
    "16",
    "32",
    "64",
    "128",
    "256"
])


ax1.set_xlabel(
    "Candidate-pool size, $M$"
)

ax1.set_ylabel(
    "Median matched-group runtime (ms)"
)

ax1.set_title(
    "(a) Runtime scaling",
    loc="left",
    fontweight="bold"
)


# Actual-value labels
runtime_offsets = [
    (4, 5),
    (4, 5),
    (4, 5),
    (4, 5),
    (-4, 7),
]

for xi, yi, off in zip(
    x,
    y_runtime,
    runtime_offsets
):

    ax1.annotate(
        f"{yi:.2f}",
        (xi, yi),
        xytext=off,
        textcoords="offset points",
        ha="left" if off[0] >= 0 else "right",
        va="bottom",
        fontsize=7.2
    )


# Fit-quality annotation
ax1.text(
    0.965,
    0.06,
    (
        rf"$\alpha={alpha:.4f}$"
        "\n"
        rf"$R^2={r_squared:.4f}$"
        "\n"
        rf"$2^\alpha={doubling_factor:.3f}$"
    ),
    transform=ax1.transAxes,
    ha="right",
    va="bottom",
    fontsize=7.8,
    bbox=dict(
        boxstyle="square,pad=0.3",
        facecolor="white",
        edgecolor="0.75",
        linewidth=0.6
    )
)


ax1.legend(
    loc="upper left",
    frameon=True,
    framealpha=1.0,
    edgecolor="0.75",
    fancybox=False
)


# Grid on major ticks only
ax1.grid(
    which="major",
    linestyle=":",
    linewidth=0.55,
    alpha=0.45
)


# ============================================================
# PANEL (b): SOLUTION COMPACTNESS
# ============================================================

selected_count_mean = (
    summary["mean_selected_count"]
    .to_numpy()
)

selected_fraction_mean = (
    summary["mean_selected_fraction_pct"]
    .to_numpy()
)


# Left axis = selected count
line_count = ax2.plot(
    x,
    selected_count_mean,
    marker="o",
    markersize=5.5,
    linewidth=1.5,
    color="#315B73",
    label="Mean selected count",
    zorder=4
)


ax2.set_xscale(
    "log",
    base=2
)

ax2.set_xticks(
    expected_sizes
)

ax2.set_xticklabels([
    "16",
    "32",
    "64",
    "128",
    "256"
])


ax2.set_xlabel(
    "Candidate-pool size, $M$"
)

ax2.set_ylabel(
    "Mean selected count"
)

ax2.set_ylim(
    3.6,
    4.9
)

ax2.set_title(
    "(b) Solution compactness",
    loc="left",
    fontweight="bold"
)


# Right axis = selected fraction
ax2_right = ax2.twinx()


line_fraction = ax2_right.plot(
    x,
    selected_fraction_mean,
    marker="s",
    markersize=5.0,
    linewidth=1.4,
    linestyle="--",
    color="#9A5A45",
    label="Mean selected fraction",
    zorder=4
)


ax2_right.set_ylabel(
    "Mean selected fraction (%)"
)

ax2_right.set_ylim(
    0,
    27
)


# ------------------------------------------------------------
# 20. LABEL PANEL (b) VALUES
# ------------------------------------------------------------

for xi, yi in zip(
    x,
    selected_count_mean
):

    ax2.annotate(
        f"{yi:.2f}",
        (xi, yi),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=7.2
    )


for xi, yi in zip(
    x,
    selected_fraction_mean
):

    ax2_right.annotate(
        f"{yi:.1f}%",
        (xi, yi),
        xytext=(0, -11),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=7.2
    )


# ------------------------------------------------------------
# 21. COMBINED LEGEND FOR PANEL (b)
# ------------------------------------------------------------

lines = (
    line_count
    + line_fraction
)

labels = [
    line.get_label()
    for line in lines
]

ax2.legend(
    lines,
    labels,
    loc="center right",
    frameon=True,
    framealpha=1.0,
    edgecolor="0.75",
    fancybox=False
)


# ------------------------------------------------------------
# 22. COMMON FORMATTING
# ------------------------------------------------------------

for ax in [ax1, ax2]:

    ax.spines["top"].set_visible(False)

    ax.tick_params(
        which="both",
        direction="in"
    )


ax2_right.spines["top"].set_visible(False)

ax2_right.tick_params(
    which="both",
    direction="in"
)


ax2.grid(
    axis="y",
    linestyle=":",
    linewidth=0.55,
    alpha=0.45
)


# ------------------------------------------------------------
# 23. SAMPLE-SIZE ANNOTATIONS
# ------------------------------------------------------------

ax1.text(
    0.98,
    0.97,
    "$100$ matched groups per $M$",
    transform=ax1.transAxes,
    ha="right",
    va="top",
    fontsize=7.7
)

ax2.text(
    0.98,
    0.97,
    "$4{,}800$ conditions per $M$",
    transform=ax2.transAxes,
    ha="right",
    va="top",
    fontsize=7.7
)


# ------------------------------------------------------------
# 24. FINAL LAYOUT
# ------------------------------------------------------------

fig.subplots_adjust(
    left=0.085,
    right=0.91,
    bottom=0.18,
    top=0.89,
    wspace=0.34
)


# ------------------------------------------------------------
# 25. EXPORT
# ------------------------------------------------------------

fig.savefig(
    OUTPUT_PDF,
    format="pdf",
    bbox_inches="tight"
)

fig.savefig(
    OUTPUT_PNG,
    format="png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


print("\nFigure successfully saved:")
print("PDF :", OUTPUT_PDF)
print("PNG :", OUTPUT_PNG)